In [2]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!


In [6]:
def test_data_loader():
    print("==================================================")
    print("🚀 QuantDataLoader 테스트를 시작합니다...")
    print("==================================================\n")
    
    # 1. 로더 인스턴스 생성
    try:
        print("[테스트 1] 로더 인스턴스화 및 환경변수 확인")
        loader = QuantDataLoader(use_cache=True)
        print("✅ 성공: DART API 키 및 로더 초기화 완료!\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")
        return

    # 2. 유니버스 로드 테스트 (Point-in-Time)
    test_date = date(2023, 7, 24)
    print(f"[테스트 2] KOSPI 유니버스 데이터 로드 ({test_date})")
    try:
        universe_df = loader.get_kospi_universe(test_date)
        print(f"✅ 성공: 총 {len(universe_df)}개 종목 로드 완료!")
        print("-" * 50)
        display(universe_df.head())
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    # 3. DART 재무제표 파싱 테스트
    ticker_to_test = '005930'
    target_year = 2023
    print(f"[테스트 3] {ticker_to_test} {target_year}년 사업보고서(11011) 파싱")
    try:
        financials = loader.parse_standardized_financials(ticker_to_test, target_year, '11011')
        print(f"✅ 성공: 재무 데이터 표준화 완료!")
        print("-" * 50)
        for key, value in financials.items():
            if pd.isna(value):
                print(f"{key:>20} : NaN")
            else:
                print(f"{key:>20} : {value:,.0f}")
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    print("==================================================")
    print("🎯 모든 테스트가 종료되었습니다.")
    print("==================================================")

# 테스트 실행
test_data_loader()

🚀 QuantDataLoader 테스트를 시작합니다...

[테스트 1] 로더 인스턴스화 및 환경변수 확인
✅ 성공: DART API 키 및 로더 초기화 완료!

[테스트 2] KOSPI 유니버스 데이터 로드 (2023-07-24)
✅ 성공: 총 834개 종목 로드 완료!
--------------------------------------------------


,ticker,name,sector,close_price,market_cap
0,005930,삼성전자,통신 및 방송 장비 제조업,70400,420272691520000
1,373220,LG에너지솔루션,일차전지 및 이차전지 제조업,597000,139698000000000
2,000660,SK하이닉스,반도체 제조업,114000,82992269610000
3,005490,POSCO홀딩스,1차 철강 제조업,642000,54294729660000
4,207940,삼성바이오로직스,기초 의약물질 제조업,742000,52811108000000


--------------------------------------------------

[테스트 3] 005930 2023년 사업보고서(11011) 파싱
✅ 성공: 재무 데이터 표준화 완료!
--------------------------------------------------
             revenue : 258,935,494,000,000
                cogs : 180,388,580,000,000
        gross_profit : 78,546,914,000,000
                 sga : 71,979,938,000,000
           inventory : 51,625,874,000,000
    operating_income : 6,566,976,000,000
          net_income : 15,487,100,000,000
 operating_cash_flow : 44,137,427,000,000
--------------------------------------------------

🎯 모든 테스트가 종료되었습니다.


In [7]:
# config.json 파라미터 불러오기
try:
    with open('config.json', 'r', encoding='utf-8') as f:
        config = json.load(f)
    
    target_market = config['strategy_params']['market'] # 예: 'KOSPI'
    top_n = config['strategy_params']['top_n_mcap']
except FileNotFoundError:
    print("⚠️ config.json 파일이 없습니다. 기본값으로 진행합니다.")
    target_market = 'KOSPI'
    top_n = 10

print(f"🔍 {target_market} 시장 데이터를 불러오는 중...\n")

# FinanceDataReader를 통한 KRX 전종목 리스팅 조회
df_krx = fdr.StockListing('KRX')

# 지정한 시장 필터링 및 시가총액(MarCap) 기준 정렬
top_mcap_df = df_krx[df_krx['Market'] == target_market].sort_values(by='Marcap', ascending=False).head(top_n)

# 보기 좋게 컬럼명 정리
top_mcap_df = top_mcap_df[['Code', 'Name', 'Close', 'Marcap', 'Stocks']].rename(
    columns={
        'Code': '종목코드',
        'Name': '종목명',
        'Close': '종가',
        'Marcap': '시가총액',
        'Stocks': '상장주식수'
    }
)

print("✅ 정상적으로 데이터를 불러왔습니다!")
display(top_mcap_df)

🔍 KOSPI 시장 데이터를 불러오는 중...

✅ 정상적으로 데이터를 불러왔습니다!


,종목코드,종목명,종가,시가총액,상장주식수
0,005930,삼성전자,254000,1484954766432000,5846278608
1,000660,SK하이닉스,1816000,1294267494840000,712702365
2,402340,SK스퀘어,1096000,144626391056000,131958386
3,005935,삼성전자우,179800,144266342299400,802371203
4,009150,삼성전기,1325000,98969147200000,74693696
5,005380,현대차,403000,82517379698000,204757766
6,373220,LG에너지솔루션,333000,77922000000000,234000000
7,207940,삼성바이오로직스,1545000,71519519295000,46290951
8,032830,삼성생명,311500,62300000000000,200000000
9,105560,KB금융,172600,61219102888400,354687734
